# RetailPulse -- Week 2 MLflow Checkpoint

**Objective:** Log Week 2 experiments and generate consolidated summary.

In [1]:
import os, warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import mlflow
warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
FIGURES_DIR = os.path.join("..", "reports", "figures")
PROCESSED_DIR = os.path.join("..", "data", "processed")
def save_fig(fig, name):
    fig.savefig(os.path.join(FIGURES_DIR, name), dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig); print(f"Saved: {name}")


In [2]:
mlflow.set_tracking_uri(os.path.join("..", "mlflow", "mlruns"))
mlflow.set_experiment("RetailPulse-Week2")

# Log churn model
churn = pd.read_csv(os.path.join(PROCESSED_DIR, "customer_churn.csv"))
with mlflow.start_run(run_name="xgboost_churn"):
    mlflow.log_param("model", "XGBoost"); mlflow.log_param("threshold", 90)
    high_risk = (churn["churn_risk"] == "High Risk").sum()
    mlflow.log_metric("high_risk_customers", high_risk)
    mlflow.log_metric("total_customers", len(churn))
    print(f"Logged churn model: {high_risk} high-risk customers")

# Log Optuna tuning
optuna_params = pd.read_csv(os.path.join(PROCESSED_DIR, "optuna_best_params.csv"))
with mlflow.start_run(run_name="optuna_tuning"):
    for _, row in optuna_params.iterrows():
        mlflow.log_param(row["param"], row["value"])
    print(f"Logged Optuna best params")

# Log drift detection
drift = pd.read_csv(os.path.join(PROCESSED_DIR, "drift_report.csv"))
with mlflow.start_run(run_name="drift_detection"):
    for _, row in drift.iterrows():
        mlflow.log_metric(f"psi_{row['Feature']}", row["PSI"])
    drifted = (drift["Status"] != "No Drift").sum()
    mlflow.log_metric("features_with_drift", drifted)
    print(f"Logged drift: {drifted} features with drift")

# Log CV results
cv = pd.read_csv(os.path.join(PROCESSED_DIR, "cv_results.csv"))
with mlflow.start_run(run_name="walk_forward_cv"):
    mlflow.log_metric("mean_mape", cv["MAPE (%)"].mean())
    mlflow.log_metric("std_mape", cv["MAPE (%)"].std())
    mlflow.log_metric("n_folds", len(cv))
    print(f"Logged CV: mean MAPE={cv['MAPE (%)'].mean():.2f}%")


Logged churn model: 2987 high-risk customers
Logged Optuna best params
Logged drift: 5 features with drift
Logged CV: mean MAPE=42.97%


In [3]:
# Week 2 summary dashboard
inv_metrics = pd.read_csv(os.path.join(PROCESSED_DIR, "inventory_metrics.csv"))
inv_sim = pd.read_csv(os.path.join(PROCESSED_DIR, "inventory_simulation.csv"), parse_dates=["Date"])
daily = pd.read_csv(os.path.join(PROCESSED_DIR, "daily_sales_features.csv"), parse_dates=["Date"])

fig, axes = plt.subplots(2, 3, figsize=(22, 12))

# Churn distribution
risk_counts = churn["churn_risk"].value_counts()
colors = {"Low Risk": "#27ae60", "Medium Risk": "#f39c12", "High Risk": "#e74c3c"}
axes[0,0].bar(risk_counts.index, risk_counts.values, color=[colors.get(r, "#3498db") for r in risk_counts.index])
axes[0,0].set_title("Churn Risk Distribution"); axes[0,0].set_ylabel("Customers")

# PSI scores
axes[0,1].barh(drift["Feature"], drift["PSI"], color=["#e74c3c" if s!="No Drift" else "#27ae60" for s in drift["Status"]])
axes[0,1].set_xlabel("PSI"); axes[0,1].set_title("Data Drift (PSI)")
axes[0,1].axvline(0.1, color="gray", linestyle="--", alpha=0.5)

# CV results
axes[0,2].bar(cv["Fold"], cv["MAPE (%)"], color="#3498db")
axes[0,2].axhline(cv["MAPE (%)"].mean(), color="#e74c3c", linestyle="--")
axes[0,2].set_title(f"Walk-Forward CV (Avg MAPE={cv['MAPE (%)'].mean():.1f}%)")
axes[0,2].set_xlabel("Fold"); axes[0,2].set_ylabel("MAPE (%)")

# Inventory simulation
axes[1,0].plot(inv_sim["Date"], inv_sim["stock_level"], color="#3498db", linewidth=1)
axes[1,0].set_title("Inventory Levels"); axes[1,0].set_ylabel("Units")

# Churn probability histogram
axes[1,1].hist(churn["churn_probability"], bins=30, color="#9b59b6", edgecolor="white")
axes[1,1].set_title("Churn Probability Distribution"); axes[1,1].set_xlabel("Probability")

# Demand trend
axes[1,2].plot(daily["Date"], daily["total_revenue"].rolling(14).mean(), color="#27ae60", linewidth=2)
axes[1,2].set_title("Revenue (14-day MA)"); axes[1,2].set_ylabel("Revenue")

fig.suptitle("RetailPulse -- Week 2 Summary", fontsize=18, fontweight="bold", y=1.01)
fig.tight_layout(); save_fig(fig, "40_week2_summary.png"); plt.show()


Saved: 40_week2_summary.png


In [4]:
print("WEEK 2 CHECKPOINT COMPLETE")
print("=" * 55)
print(f"Churn model: {len(churn)} customers scored")
print(f"Optuna: 50 trials completed")
print(f"Drift: {drifted} features with detected drift")
print(f"CV: {len(cv)} folds, mean MAPE = {cv['MAPE (%)'].mean():.2f}%")


WEEK 2 CHECKPOINT COMPLETE
Churn model: 5878 customers scored
Optuna: 50 trials completed
Drift: 5 features with detected drift
CV: 18 folds, mean MAPE = 42.97%
